# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj), which is published as a Croissant package.

Using the `mlcroissant` library, we'll walk through accessing the metadata, loading data records, performing initial data analysis, and visualization — **by referencing all data entities with their Croissant `@id`** as per FAIR best practices.

### Dataset Source
The dataset source is defined via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install required library
!pip install mlcroissant

## 1. Data Loading
We load the Croissant dataset and display high-level metadata using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Already a Croissant Metadata object (not a dict)

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's enumerate the available record sets and their fields as defined in the dataset's Croissant metadata. All references use Croissant `@id` as required, ensuring fidelity with the structured data model.

Note: The dataset includes multiple record sets and fields that can be programmatically walked using the `dataset.list_record_sets()` and `dataset.list_fields()` methods.

In [ ]:
# List all record sets by their Croissant @id
print("Available Record Sets (`@id`):")
rec_set_ids = dataset.list_record_sets()
for rec_set_id in rec_set_ids:
    print(f"  - {rec_set_id}")

# List fields for each record set
for rec_set_id in rec_set_ids:
    print(f"\nFields for Record Set: {rec_set_id}")
    field_ids = dataset.list_fields(rec_set_id)
    for field_id in field_ids:
        field = dataset.get_field(rec_set_id, field_id)
        print(f"  - @id: {field_id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
We can load all data from each record set into Pandas DataFrames, referencing the record set and field `@id` values as found above. For demonstration, we'll show the record set `@id` and the fields in use, and extract the data accordingly.

> *Choose relevant record sets and adjust the list below as the dataset evolves, by updating with their `@id`s.*

In [ ]:
# List of all primary record sets in the dataset
record_sets = dataset.list_record_sets()

dataframes = {}
for rec_set_id in record_sets:
    print(f"Loading record set: {rec_set_id}")
    # These will be dictionaries with field @ids as keys
    recs = list(dataset.records(record_set=rec_set_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rec_set_id] = df
        print(f"  Columns (@id): {df.columns.tolist()}")
        display(df.head(2))  # Show a preview
    else:
        print('  No records found.')

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA on the primary survey record set. We will reference columns (fields) using their `@id` (not just human names). You can adjust numeric fields or grouping fields based on the output in step 3.

Examples shown below use typical fields such as scores from the survey (e.g., PHQ-9, GAD-7, PSQ) — but always using the Croissant `@id` as the column name.

In [ ]:
# Example: Choose the main survey record set (replace with your record_set_id)
main_survey_set_id = None
for rsid in dataframes.keys():
    # Heuristic: find the largest DataFrame, likely to be the main table
    if main_survey_set_id is None or dataframes[rsid].shape[0] > dataframes[main_survey_set_id].shape[0]:
        main_survey_set_id = rsid

df = dataframes[main_survey_set_id]
print(f"Analyzing record set: {main_survey_set_id} (rows: {df.shape[0]})")

# View column @ids and try to pick a numeric field (example: PHQ-9 score, GAD-7 score, or PSQ score)
print("Available columns (field @ids):", df.columns.tolist())

# Choose a numeric field for thresholding; update with the actual @id as needed
# Let's heuristically pick a column with 'phq' or 'gad' or 'psq' in its @id/text
field_choices = [col for col in df.columns if any(k in col.lower() for k in ['phq', 'gad', 'psq'])]
if field_choices:
    numeric_field_id = field_choices[0]
else:
    # Default fallback: first numeric-like column
    numeric_field_id = df.select_dtypes(include=[np.number]).columns[0]

print(f"Selected numeric field: {numeric_field_id}")
threshold = 5  # Domain-appropriate threshold (e.g., minimal symptoms)

# Filtering the records
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} rows")
display(filtered_df.head())

# Normalize the selected numeric field
col_normalized = numeric_field_id + '_normalized'
filtered_df[col_normalized] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, col_normalized]].head())

# Try grouping by a demographic field (e.g., gender or education); update with actual field @id
group_field_candidates = [c for c in df.columns if any(w in c.lower() for w in ['gender', 'age', 'village', 'education'])]
if group_field_candidates:
    group_field = group_field_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    print("No groupable demographic field found.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field (e.g., PHQ-9/GAD-7/PSQ score), and (if available) show differences across demographic groupings.

We always reference columns by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of scores
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_field_candidates:
    plt.figure(figsize=(10,5))
    group_field = group_field_candidates[0]
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library, **referencing all data entities by their Croissant `@id`**.

We:
- Loaded and previewed metadata
- Discovered the structure (record sets, fields/columns)
- Extracted tabular data for analysis
- Filtered and normalized survey scores
- Visualized important trends

**This workflow can easily be adapted to any Croissant-compliant dataset by following the pattern of using API methods and always referencing structural elements by `@id`.**